In [7]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

### Cek GPU

In [8]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")
if device.type == 'cuda':
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")

Menggunakan device: cuda:0
Nama GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


### Hyperparameter & Data Loading (Train/Validation Split)

In [9]:
BATCH_SIZE = 32  
EPOCHS = 10
LEARNING_RATE = 1e-3
IMG_SIZE = 224

data_dir = '../Data/train_cropped/' 

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=val_transform)
class_names = full_dataset.classes
print(f"Daftar Kelas: {class_names}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Jumlah Data Training: {len(train_dataset)}")
print(f"Jumlah Data Validation: {len(val_dataset)}")

Daftar Kelas: ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
Jumlah Data Training: 1152
Jumlah Data Validation: 289


### Inisialisasi Model, Loss, dan Optimizer

In [10]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Model, Loss Function, dan Optimizer siap!")

Model, Loss Function, dan Optimizer siap!


### Training Loop

In [11]:
train_losses, val_losses, val_accuracies = [], [], []
best_val_acc = 0.0

print("Mulai proses training")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad() 
        outputs = model(inputs) 
        loss = criterion(outputs, labels) 
        loss.backward() 
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        train_pbar.set_postfix({'loss': loss.item()})
        
    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    model.eval() 
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]')
        for inputs, labels in val_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct / total
    
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)
    
    print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), 'best_baseline_model.pth')
        print(f"Model terbaik baru disimpan (Akurasi: {best_val_acc:.4f})")

Mulai proses training


Epoch 1/10 [Train]:   0%|          | 0/36 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 1.5281 | Val Loss: 0.9557 | Val Acc: 0.7647
Model terbaik baru disimpan (Akurasi: 0.7647)


Epoch 2/10 [Train]:   0%|          | 0/36 [00:00<?, ?it/s]

...

Epoch 9/10 [Val]:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 9 | Train Loss: 0.1768 | Val Loss: 0.8115 | Val Acc: 0.8581


Epoch 10/10 [Train]:   0%|          | 0/36 [00:00<?, ?it/s]

Epoch 10/10 [Val]:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 10 | Train Loss: 0.2312 | Val Loss: 0.8878 | Val Acc: 0.8304
